In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)


In [2]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

#region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])

map.centerObject(point, 10)
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')

map #just testing if everything is working

Map(center=[23.8041, 90.4152], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topri…

In [15]:
sar_collection = ee.ImageCollection('COPERNICUS/S1_GRD') \
    .filterBounds(region) \
    .filterDate('2024-07-01', '2024-08-30') \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
    .filter(ee.Filter.eq('instrumentMode', 'IW'))

#vv means vertical transmit and vertical receive, it provides information about surface roughness
#vh means vertical transmit and horizontal receive, it provides information about vegetation structure
#iw instrument mode is the most commonly used mode for land applications, providing a good balance between spatial resolution and coverage area.


non_sar_collection_at_the_same_time = ee.ImageCollection('COPERNICUS/S2_SR') \
    .filterBounds(region) \
    .filterDate('2024-07-01', '2024-08-30') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 100))


In [16]:
sar_image = sar_collection.median().clip(region)
non_sar_image = non_sar_collection_at_the_same_time.median().clip(region)

In [18]:
print('radar image found during monsoon:', sar_collection.size().getInfo())
print('non radar image found during monsoon:', non_sar_collection_at_the_same_time.size().getInfo())

for img in non_sar_collection_at_the_same_time.toList(non_sar_collection_at_the_same_time.size()).getInfo():
    print("cloud coverage of non_sar image:", img['properties']['CLOUDY_PIXEL_PERCENTAGE'])

radar image found during monsoon: 7
non radar image found during monsoon: 20
cloud coverage of non_sar image: 99.516153
cloud coverage of non_sar image: 99.51666
cloud coverage of non_sar image: 88.959694
cloud coverage of non_sar image: 88.690269
cloud coverage of non_sar image: 78.257674
cloud coverage of non_sar image: 79.797804
cloud coverage of non_sar image: 99.716657
cloud coverage of non_sar image: 99.75844
cloud coverage of non_sar image: 93.208516
cloud coverage of non_sar image: 90.893936
cloud coverage of non_sar image: 99.75419
cloud coverage of non_sar image: 99.786621
cloud coverage of non_sar image: 94.987249
cloud coverage of non_sar image: 95.359987
cloud coverage of non_sar image: 92.076588
cloud coverage of non_sar image: 92.292631
cloud coverage of non_sar image: 99.500597
cloud coverage of non_sar image: 99.65294
cloud coverage of non_sar image: 99.997854
cloud coverage of non_sar image: 99.995118


As we can see, during that time there was very high cloud and with even 50% cloud coverage, we can't find any image. 

In [19]:
vis_params = {
    'bands': ['VV', 'VH', 'VV'], # mapping bands to R, G, B channels
    'min': -25,
    'max': -5,
}

radar_image = geemap.Map(center = region.centroid().coordinates().getInfo()[::-1], zoom = 10)
radar_image.addLayer(sar_image, vis_params, 'Sentinel-1 Radar')
radar_image

Map(center=[23.800006561617693, 90.39999999999813], controls=(WidgetControl(options=['position', 'transparent_…

In [21]:
#now non-sar image visualization

non_sar_vis_params = {
    'bands': ['B4', 'B3', 'B2'], # RGB bands for Sentinel-2
    'min': 0,
    'max': 3000,
}

non_radar_image = geemap.Map(center = region.centroid().coordinates().getInfo()[::-1], zoom = 10)
non_radar_image.addLayer(non_sar_image, non_sar_vis_params, 'Sentinel-2 RGB')

non_radar_image


Map(center=[23.800006561617693, 90.39999999999813], controls=(WidgetControl(options=['position', 'transparent_…